# Exercise 4 — run_daily_loop and _sleep_fn injection

`run_daily_loop` is the production scheduler: it runs forever (or N times), waiting for the daily run_time between iterations. The `_sleep_fn` injection parameter makes it testable without any actual waiting — a lambda that does nothing replaces time.sleep. This is the same injection pattern used throughout the course.

In [ ]:
import pandas as pd, math, datetime, pathlib, tempfile

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

def next_run_time(run_time_str="16:00"):
    now = datetime.datetime.now()
    h, m = (int(x) for x in run_time_str.split(":"))
    target = now.replace(hour=h, minute=m, second=0, microsecond=0)
    if target <= now: target += datetime.timedelta(days=1)
    return target

def seconds_until(target_dt):
    delta = target_dt - datetime.datetime.now()
    return max(0.0, delta.total_seconds())


def run_daily_loop(bot_fn, run_time="16:00", max_iterations=None,
                   _sleep_fn=None):
    """Run bot_fn once per day at run_time.

    Algorithm:
      if _sleep_fn is None: _sleep_fn = time.sleep
      count = 0
      while max_iterations is None or count < max_iterations:
          _sleep_fn(seconds_until(next_run_time(run_time)))
          bot_fn()
          count += 1
      return count

    Returns:
        int — number of times bot_fn was called
    """
    import time
    if _sleep_fn is None: _sleep_fn = time.sleep
    # TODO: ~5 lines
    return 0


### Checks

In [ ]:
checks = 0
NO_SLEEP = lambda s: None   # skip all waits

# 1 — max_iterations=1 calls bot_fn once
try:
    calls = []
    count = run_daily_loop(lambda: calls.append(1), max_iterations=1,
                           _sleep_fn=NO_SLEEP)
    assert len(calls) == 1, f"expected 1 call, got {len(calls)}"
    assert count == 1
    checks += 1; print("✅ 1 max_iterations=1 → bot_fn called once, returns 1")
except Exception as e:
    print("❌ 1:", e)

# 2 — max_iterations=5 calls bot_fn five times
try:
    calls = []
    count = run_daily_loop(lambda: calls.append(1), max_iterations=5,
                           _sleep_fn=NO_SLEEP)
    assert len(calls) == 5, f"expected 5 calls, got {len(calls)}"
    assert count == 5
    checks += 1; print("✅ 2 max_iterations=5 → bot_fn called 5 times, returns 5")
except Exception as e:
    print("❌ 2:", e)

# 3 — _sleep_fn is called with a non-negative number
try:
    slept = []
    run_daily_loop(lambda: None, max_iterations=3,
                   _sleep_fn=lambda s: slept.append(s))
    assert len(slept) == 3
    assert all(s >= 0 for s in slept), f"sleep times should be ≥ 0, got {slept}"
    checks += 1; print(f"✅ 3 _sleep_fn called 3× with non-negative seconds: {[round(s,1) for s in slept]}")
except Exception as e:
    print("❌ 3:", e)

# 4 — bot_fn exception propagates (loop does not swallow errors)
try:
    def failing_fn():
        raise ValueError("intentional error")
    try:
        run_daily_loop(failing_fn, max_iterations=1, _sleep_fn=NO_SLEEP)
        print("❌ 4: expected ValueError to propagate")
    except ValueError as ve:
        assert "intentional error" in str(ve)
        checks += 1; print("✅ 4 bot_fn exceptions propagate out of run_daily_loop")
except Exception as e:
    print("❌ 4:", e)

# 5 — max_iterations=0 calls bot_fn zero times
try:
    calls = []
    count = run_daily_loop(lambda: calls.append(1), max_iterations=0,
                           _sleep_fn=NO_SLEEP)
    assert len(calls) == 0 and count == 0
    checks += 1; print("✅ 5 max_iterations=0 → bot_fn never called, returns 0")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
